# 04 · Incerteza e abstenção

**Bloco 4 do workshop.**

Pergunta: *o modelo sabe quando não sabe?*

O modelo não precisa responder tudo. Ele pode devolver os casos mais incertos para revisão humana.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# dataset canônico do workshop — NÃO altere estes parâmetros
X, y = make_classification(n_samples=20000, n_features=20, n_informative=8,
                           n_redundant=4, weights=[0.99, 0.01], flip_y=0.0,
                           class_sep=1.5, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
# split extra para calibrar no notebook 03 (nunca calibre no teste)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.25,
                                              stratify=y_tr, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f"teste: {len(y_te)} casos | {int(y_te.sum())} fraudes | prevalência {y_te.mean():.2%}")

## 4.1 Recuperar o limiar operacional do notebook 02

In [ ]:
C_FP, C_FN = 5, 500
ths    = np.linspace(0.001, 0.999, 999)
custos = np.array([C_FP*int(((proba >= t) & (y_te == 0)).sum())
                 + C_FN*int(((proba <  t) & (y_te == 1)).sum()) for t in ths])
t_ot   = float(ths[int(custos.argmin())])
pred_t = (proba >= t_ot).astype(int)
print(f"limiar operacional: {t_ot:.4f}")

## 4.2 Confiança = distância ao limiar operacional

⚠️ **Cuidado com o erro mais comum aqui.** Usar `np.maximum(proba, 1-proba)` mede a distância ao 0,5, não ao limiar que você está usando de verdade. Com t* = 0,041, casos com probabilidade perto de zero receberiam confiança perto de 1, quando na verdade estão **próximos da fronteira de decisão**.

Segundo detalhe: `np.quantile` sobre a margem falha quando há muitos scores empatados, e a Random Forest produz muitos zeros exatos. Selecione por `np.argsort`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

margem = np.abs(proba - t_ot)
ordem  = np.argsort(margem)        # menor margem = mais incerto
n      = len(y_te)

print("cobertura | precisão | recall | enviados | fraudes entre os enviados")
for cov in [1.0, 0.98, 0.95, 0.90, 0.80]:
    k = int(round(n*(1-cov)))
    keep = np.ones(n, bool); keep[ordem[:k]] = False
    print(f"{cov:>8.0%}  |  {precision_score(y_te[keep], pred_t[keep], zero_division=0):.4f}"
          f"  |  {recall_score(y_te[keep], pred_t[keep]):.4f}"
          f"  |  {(~keep).sum():>7}  |  {int(y_te[~keep].sum()):>3}")

## 4.3 Curva risco-cobertura

In [ ]:
import matplotlib.pyplot as plt

covs, precs, recs = [], [], []
for cov in np.linspace(0.6, 1.0, 21):
    k = int(round(n*(1-cov)))
    keep = np.ones(n, bool); keep[ordem[:k]] = False
    covs.append(cov)
    precs.append(precision_score(y_te[keep], pred_t[keep], zero_division=0))
    recs.append(recall_score(y_te[keep], pred_t[keep]))

plt.figure(figsize=(8, 4))
plt.plot(covs, precs, 'o-', label='precisão')
plt.plot(covs, recs,  'o-', label='recall')
plt.xlabel('cobertura (fração respondida pelo modelo)'); plt.ylim(0, 1)
plt.legend(); plt.tight_layout(); plt.show()

### ✏️ Tarefa — a política de abstenção do seu modelo

Não basta escolher a cobertura. Responda as quatro:

1. Qual cobertura, e qual o ganho de precisão e recall?
2. Quantos casos por dia isso envia para revisão? **Número absoluto, não porcentagem.**
3. Quem revisa, e essa capacidade existe hoje?
4. Quantos positivos reais ficam entre os abstidos, e qual o custo do atraso?

In [ ]:
cobertura_escolhida  = None    # ex: 0.90
volume_diario        = None    # casos/dia enviados a humano
quem_revisa          = "..."
custo_do_atraso      = "..."

print(f"cobertura {cobertura_escolhida} | {volume_diario} casos/dia | revisor: {quem_revisa}")